Cell 1: Setup

In [1]:
# Cell 1: Setup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, subprocess, sys
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = f'https://{TOKEN}@github.com/RidwanHR5/moshilite.git'

if not os.path.exists('/content/moshilite'):
    subprocess.run(['git', 'clone', REPO, '/content/moshilite'], check=True)
else:
    subprocess.run(['git', '-C', '/content/moshilite', 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/moshilite', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'scipy', 'moshi', 'pyyaml', 'scikit-learn'], check=True)
print('✅ Setup complete')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete


Cell 2: Check token status



In [2]:
# Cell 2: Verify encoded tokens are available
!python /content/moshilite/scripts/download_datasets.py --status



=== Raw datasets ===
  ✅ Moshi weights: 11 files
  ❌ LibriSpeech (raw): not found
  ❌ LibriLight small (raw): not found
  ❌ AMI (raw): not found

=== Encoded tokens on GDrive ===
  ✅ Tokens: LibriSpeech: 138120 files, 474.7 hrs, 8 shards (4 stats files)
  ✅ Tokens: LibriLight: 2588 files, 577.2 hrs, 7 shards (1 stats files)
  ✅ Tokens: AMI: 108502 files, 77.9 hrs, 1 shards (1 stats files)
  ✅ Tokens: LibriMix: 14269 files, 57.5 hrs, 1 shards (1 stats files)


Cell 3: Load Mimi encoder → Load Moshi model



In [2]:
# Cell 3: Load Moshi for analysis (INT4 — ~3.8 GB VRAM)
import logging
logging.basicConfig(level=logging.INFO)

from moshilite.analysis.moshi_model import load_moshi_for_analysis, get_model_info

# If weights are on GDrive (Account 1):
model = load_moshi_for_analysis(
    weights_dir='/content/drive/MyDrive/moshilite/upstream_weights/moshiko',
    precision='int4',
    device='cuda',
)
print(get_model_info(model))


{'total_params': 7689678080, 'total_params_b': 7.68967808, 'd_model': 4096, 'dep_q': 8, 'n_q': 16, 'n_layers': 32, 'n_heads': 32, 'd_ffn': 11264}


Git Pull

In [ ]:
!cd /content/moshilite && git pull && pip install -e . --force-reinstall --no-deps -q

Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for moshilite (pyproject.toml) ... done


Cell 4: Load calibration data from pre-encoded tokens



In [3]:
# Cell 4: Load calibration batches
from moshilite.analysis.moshi_hooks import prepare_token_batches, get_n_layers, get_n_heads

# Single-speaker data (LibriSpeech)
single_data = prepare_token_batches(
    '/content/drive/MyDrive/moshilite/tokens/librispeech',
    n_samples=200, seq_len=500, n_codebooks=17, batch_size=4,
)

# Dialogue data (LibriMix)
dialogue_data = prepare_token_batches(
    '/content/drive/MyDrive/moshilite/tokens/librimix',
    n_samples=200, seq_len=500, n_codebooks=17, batch_size=4,
)

print(f'✅ Loaded {len(single_data)} single-speaker batches, {len(dialogue_data)} dialogue batches')
print(f'   Shape: {single_data[0].shape}')


✅ Loaded 50 single-speaker batches, 50 dialogue batches
   Shape: torch.Size([4, 17, 500])


Cell 5: Compute BI scores + DSR tags



In [4]:
# Cell 5: Block Influence scores + Dual-Stream criticality tagging
from moshilite.analysis.block_influence import compute_bi_scores
from moshilite.analysis.dual_stream import tag_layers, save_layer_tags
from moshilite.analysis.moshi_hooks import moshi_layer_hook_fn

print('📈 Computing BI scores (single-speaker)...')
bi_single = compute_bi_scores(model, single_data, moshi_layer_hook_fn, device='cuda')

print('📈 Computing BI scores (dialogue)...')
bi_dialogue = compute_bi_scores(model, dialogue_data, moshi_layer_hook_fn, device='cuda')

# DSR tagging with v8 thresholds
thresholds = {
    'dual_stream_critical_dsr': 2.0,
    'dual_stream_critical_bi_top_pct': 0.20,
    'dialogue_critical_dsr': 1.5,
    'dialogue_critical_bi_top_pct': 0.30,
    'fallback_critical_pct': 0.70,
}

dsr_result = tag_layers(bi_single, bi_dialogue, thresholds)

# Save
out_dir = '/content/drive/MyDrive/moshilite/eval/prune30_v1/stage1_analysis'
!mkdir -p {out_dir}
save_layer_tags(dsr_result, f'{out_dir}/layer_tags.json')

print(f'\n📊 DSR Summary: {dsr_result.summary()}')
print(f'🟢 Pruneable layers: {dsr_result.get_pruneable_layers()}')
print(f'🔴 Critical layers: {dsr_result.get_critical_layers()}')


📈 Computing BI scores (single-speaker)...
📈 Computing BI scores (dialogue)...
✅ Layer tags saved to /content/drive/MyDrive/moshilite/eval/prune30_v1/stage1_analysis/layer_tags.json

📊 DSR Summary: {'n_layers': 32, 'n_general': 22, 'n_dialogue_critical': 10, 'n_dual_stream_critical': 0, 'fallback_triggered': False, 'mean_dsr': 0.9635618014715901, 'max_dsr': 1.1307700063970583}
🟢 Pruneable layers: [5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
🔴 Critical layers: [0, 1, 2, 3, 4, 7, 12, 19, 20, 31]


attribute name

In [7]:
# Run this first — inspect actual attribute names on StreamingMultiheadAttention
layer0 = model.transformer.layers[0]
attn   = layer0.self_attn

print('=== StreamingMultiheadAttention attributes ===')
print([name for name, _ in attn.named_children()])

print('\n=== Named parameters (first 20) ===')
for name, p in list(attn.named_parameters())[:20]:
    print(f'  {name:40s}  {tuple(p.shape)}')


=== StreamingMultiheadAttention attributes ===
['rope', 'out_projs', 'in_projs']

=== Named parameters (first 20) ===
  out_projs.0.weight                        (4096, 4096)
  out_projs.0.weight_scb                    (4096,)
  in_projs.0.weight                         (12288, 4096)
  in_projs.0.weight_scb                     (12288,)


Cell 6: Compute Head Importance


In [8]:
# Cell 6 (revised): Weight-magnitude head importance — no forward pass, zero OOM risk
import torch
import numpy as np
from moshilite.analysis.head_importance import HeadImportanceResult
from moshilite.analysis.moshi_hooks import get_n_layers, get_n_heads

n_layers = get_n_layers(model)  # 32
n_heads  = get_n_heads(model)   # 32

scores = np.zeros((n_layers, n_heads))

for layer_idx, layer in enumerate(model.transformer.layers):
    # Moshi uses out_projs[0] for the main transformer
    W = layer.self_attn.out_projs[0].weight.detach().float().cpu()

    d_model = W.shape[0]
    head_dim = d_model // n_heads  # 128

    # Each head contributed head_dim columns to out_proj
    for head_idx in range(n_heads):
        col_start = head_idx * head_dim
        col_end   = col_start + head_dim
        head_slice = W[:, col_start:col_end]   # [4096, 128]
        scores[layer_idx, head_idx] = head_slice.norm('fro').item()

# Build result in the same format as compute_head_importance
ranked = sorted(
    [(l, h, float(scores[l, h])) for l in range(n_layers) for h in range(n_heads)],
    key=lambda x: x[2]
)

head_result = HeadImportanceResult(importance_scores=scores, ranked_heads=ranked)

print(f'✅ Head importance computed (weight-norm proxy).')
print(f'   Score range: [{scores.min():.2f}, {scores.max():.2f}]')
print(f'   Top-5 least important heads: {head_result.get_least_important(5)}')


✅ Head importance computed (weight-norm proxy).
   Score range: [9341.86, 28799.35]
   Top-5 least important heads: [(0, 23), (0, 20), (0, 29), (0, 8), (0, 0)]


Cell 7: Compute FFN Importance


In [9]:
# Cell 7: PCA-guided FFN channel importance
from moshilite.analysis.ffn_importance import compute_ffn_importance
from moshilite.analysis.moshi_hooks import moshi_ffn_hook_fn

# Only analyze pruneable layers (no point analyzing critical layers)
pruneable_layers = dsr_result.get_pruneable_layers()
print(f'🔬 Analyzing FFN channels in {len(pruneable_layers)} pruneable layers: {pruneable_layers}')

ffn_result = compute_ffn_importance(
    model=model,
    data=single_data[:20],       # 20 batches; [B, 17, T] each
    layer_indices=pruneable_layers,
    ffn_hook_fn=moshi_ffn_hook_fn,
    device='cuda',
    n_components=64,
)

print(f'✅ FFN importance computed.')
for layer_idx in pruneable_layers[:5]:   # preview first 5
    scores = ffn_result.importance_scores[layer_idx]
    print(f'   Layer {layer_idx:2d}: score range [{scores.min():.4f}, {scores.max():.4f}]  '
          f'(d_ffn={len(scores)})')


🔬 Analyzing FFN channels in 22 pruneable layers: [5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
✅ FFN importance computed.
   Layer  5: score range [0.0000, 0.9380]  (d_ffn=4096)
   Layer  6: score range [0.0000, 0.0161]  (d_ffn=4096)
   Layer  8: score range [0.0000, 0.0073]  (d_ffn=4096)
   Layer  9: score range [0.0000, 0.0072]  (d_ffn=4096)
   Layer 10: score range [0.0000, 0.0052]  (d_ffn=4096)


Cell 8: Save All Results + Summary


In [11]:
# Cell 8: Save all analysis results
import numpy as np
import json, os

out_dir = '/content/drive/MyDrive/moshilite/eval/prune30_v1/stage1_analysis'
os.makedirs(out_dir, exist_ok=True)

# --- 1. BI scores ---
np.savez(
    f'{out_dir}/bi_scores.npz',
    bi_single=bi_single.bi_scores,
    bi_dialogue=bi_dialogue.bi_scores,
)
print(f'💾 Saved bi_scores.npz')

# --- 2. Layer tags (already saved in Cell 5) ---
tags_path = f'{out_dir}/layer_tags.json'
assert os.path.exists(tags_path), 'layer_tags.json missing — re-run Cell 5'
print(f'💾 layer_tags.json already saved ✅')

# --- 3. Head importance ---
np.savez(
    f'{out_dir}/head_importance.npz',
    importance_scores=head_result.importance_scores,   # [n_layers, n_heads]
)
print(f'💾 Saved head_importance.npz')

# --- 4. FFN importance ---
ffn_data = {f'layer_{k}': v for k, v in ffn_result.importance_scores.items()}
np.savez(f'{out_dir}/ffn_importance.npz', **ffn_data)
print(f'💾 Saved ffn_importance.npz  ({len(ffn_data)} layers)')

# --- 5. Human-readable summary JSON ---
dsr_summary = dsr_result.summary()  # has n_layers, n_general, etc.
summary = {
    'n_layers':          dsr_summary['n_layers'],
    'pruneable_layers':  dsr_result.get_pruneable_layers(),
    'critical_layers':   dsr_result.get_critical_layers(),
    'dsr_stats':         dsr_summary,
    'bi_single_per_layer':   bi_single.bi_scores.tolist(),
    'bi_dialogue_per_layer': bi_dialogue.bi_scores.tolist(),
    'head_importance_mean_per_layer': head_result.importance_scores.mean(axis=1).tolist(),
    'ffn_analyzed_layers': list(ffn_result.importance_scores.keys()),
}

with open(f'{out_dir}/stage1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'💾 Saved stage1_summary.json')

# --- Final report ---
print('\n' + '='*60)
print('✅ Stage 1 Analysis Complete')
print('='*60)
print(f"  n_layers   : {dsr_summary['n_layers']}")
print(f"  Pruneable  : {dsr_result.get_pruneable_layers()}")
print(f"  Critical   : {dsr_result.get_critical_layers()}")
print('='*60)


💾 Saved bi_scores.npz
💾 layer_tags.json already saved ✅
💾 Saved head_importance.npz
💾 Saved ffn_importance.npz  (22 layers)
💾 Saved stage1_summary.json

✅ Stage 1 Analysis Complete
  n_layers   : 32
  Pruneable  : [5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
  Critical   : [0, 1, 2, 3, 4, 7, 12, 19, 20, 31]
